# w9_viewgrid.ipynb -- VIEW-COMPOSITION grid (@512 screening, 2000ep)

Explicit view grammar (repo-root model_history.md): [d<k>][w<k>][sp<k>]r<n>_i2ce
-- d = tiered doc slot (wiki -> sp -> review fallback, the protocol slot),
w = wiki-only slot, sp = store-page-only slot (full sp coverage, wiki games
NOT excluded), r = review views. Protocol i2ce == d1r3 (NV=4).

Cells: the v_review dose ladder d1r4/d1r5/d1r6 (+1/+2/+3 review views, one
tiered doc slot kept) and w1sp1r3 (wiki AND store-page docs COEXIST in one
step -- the tiered protocol never lets a game see both). Baselines for the
readout: i2ce@512@2000ep (= d1r3, scale grid) and nodoc (4R+0D, 1000ep --
budget mismatch, noted). Per-view-sum convention: CE terms and I edges grow
with NV (4->5/6/7 terms; 6->10/15/21 edges) -- the grid scales the WHOLE
objective with the view count, not I alone.

Labels/claims are the campaign contract (same rows sit in w9_jobs FS_JOBS;
claim files make the two paths mutually exclusive -- safe to run alongside
any other pod). VRAM-scheduled per ARM (same cap, different NV -> different
peaks; warmup measures each arm's real step). AUTO-STOPS when drained.


In [ ]:
# constants
import os

REPO = os.path.abspath("..")   # this release folder (contains Pod/ and VICReg_review/)
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # shared campaign out dir

# VRAM scheduler knobs (@512 arms are light; pack many per GPU).
SAFETY = 0.85
RESERVE_GIB = 1.5

# (arm, cap, epochs) -- all @512 clean fp, 2000ep (single-budget curve,
# matches the scale grid's @512 cells; FS_JOBS carries the same rows with
# the per-job epochs-override field).
VG_EPOCHS = 2000
FLASH = [
    ("wcle_d1r4_i2ce_icetf", 512, VG_EPOCHS),     # 4R+1D  NV=5
    ("wcle_d1r5_i2ce_icetf", 512, VG_EPOCHS),     # 5R+1D  NV=6
    ("wcle_d1r6_i2ce_icetf", 512, VG_EPOCHS),     # 6R+1D  NV=7
    ("wcle_w1sp1r3_i2ce_icetf", 512, VG_EPOCHS),  # 3R+1W+1SP  NV=5
]
os.makedirs(OUT_DIR, exist_ok=True)
print(f"{len(FLASH)} view-grid cells:",
      [(a.replace('wcle_', '').replace('_icetf', ''), c) for a, c, _e in FLASH])


In [ ]:
# Local setup (release build: the code ships with this folder -- no
# repository synchronisation is needed or performed).
import importlib.util
import os
import sys
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        %pip -q install scikit-learn scipy
        break
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")

In [ ]:
# Stage the corpus into RAM (llm views not needed for this grid; the
# w1sp1r3 arm needs BOTH wiki_clean and sp_raw view packs -- both listed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- worker will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# VRAM-BUDGET DYNAMIC SCHEDULER -- per-ARM warmup (all cells share cap 512
# but NV=5/6/7 -> different step peaks: d1r6 does 7 tower forwards + 21 I
# edges per step). Warmup runs the REAL worker step per arm; jobs go to the
# least-loaded GPU that fits; an oversized job still runs SOLO (no deadlock).
import os, subprocess, tempfile, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
gpus = J.detect_gpus()

def _smi_mib(field, g):
    out = subprocess.check_output(
        ["nvidia-smi", f"--query-gpu={field}", "--format=csv,noheader,nounits",
         "-i", str(g)]).decode().strip().split("\n")[0]
    return int(out) * 2**20

free = {g: _smi_mib("memory.free", g) for g in gpus}
budget = {g: int(free[g] * SAFETY - RESERVE_GIB * 2**30) for g in gpus}
print(f"[vram] free/GPU ~{min(free.values())/2**30:.0f}G  budgets "
      f"{[f'{budget[g] / 2**30:.0f}G' for g in gpus]}")

# --- warmup: peak bytes per ARM (same cap, different view count) ---
cost = {}
for arm, cap, _e in FLASH:
    tf = Path(tempfile.gettempdir()) / f"w9vram_{arm}_g{cap}.txt"
    tf.unlink(missing_ok=True)
    cmd = ["python", "-u", J.FS_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           OUT_DIR, "--repo", REPO, "--arm", arm, "--anchor-cap", str(cap),
           "--epochs", "1", "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--measure-vram", str(tf)]
    print(f"[warmup] {arm} @g{cap} ...", flush=True)
    with open(logd / f"measure_{arm}_g{cap}.log", "w") as fh:
        subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpus[0]))
    cost[arm] = int(tf.read_text()) if tf.exists() else budget[gpus[0]] + 1
    print(f"[warmup] {arm}: {cost[arm] / 2**30:.2f}G peak", flush=True)

# --- job list (skip done), best-fit decreasing ---
todo = []
for arm, cap, ep in FLASH:
    nm = J.fs_label(arm, cap, False, 0, "clean", 16)
    if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{ep}.npz").exists():
        print(f"[skip] {nm} at {ep}"); continue
    todo.append((arm, cap, ep, nm, cost[arm]))
todo.sort(key=lambda j: -j[4])

# --- dynamic pool scheduler ---
now_used = {g: 0 for g in gpus}
fails = []
cv = threading.Condition()

def run_job(g, arm, cap, ep, nm, c):
    try:
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
        cmd = ["python", "-u", J.FS_WORKER, "--data-dir", DATA_DIR, "--out-dir",
               OUT_DIR, "--repo", REPO, "--arm", arm, "--anchor-cap", str(cap),
               "--epochs", str(ep), "--ckpt-every", str(J.CKPT_EVERY),
               "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
               "--topup-seeds", str(J.TOPUP_SEEDS),
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        t0 = time.time()
        with open(logd / f"{arm}_g{cap}.log", "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
        if p.returncode != 0:
            (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
        print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} @{ep} "
              f"[{(time.time() - t0) / 60:.1f}m]", flush=True)
    finally:
        with cv:
            now_used[g] -= c
            cv.notify_all()

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
active = []
with cv:
    pending = list(todo)
    while pending or any(t.is_alive() for t in active):
        progressed = False
        i = 0
        while i < len(pending):
            arm, cap, ep, nm, c = pending[i]
            fit = [g for g in gpus if now_used[g] + c <= budget[g] or now_used[g] == 0]
            if not fit:
                i += 1; continue
            g = min(fit, key=lambda g: now_used[g])   # least-loaded -> spread
            now_used[g] += c
            th = threading.Thread(target=run_job,
                                  args=(g, arm, cap, ep, nm, c), daemon=True)
            active.append(th); th.start(); pending.pop(i)
            print(f"[sched] {nm} -> gpu{g}  ({c / 2**30:.1f}G, used "
                  f"{now_used[g] / 2**30:.1f}/{budget[g] / 2**30:.0f}G)", flush=True)
            progressed = True
        active = [t for t in active if t.is_alive()]
        if not progressed:
            cv.wait(timeout=3)
stop_evt.set()
for t in active:
    t.join()
print(f"drained; {len(fails)} failed")
for nm in fails:
    print("  FAILED:", nm)


In [ ]:
# Readout: the view-composition grid, ZSbest-primary (val-selected ZS).
# Dose curve: d1r3(=i2ce@512) -> d1r4 -> d1r5 -> d1r6; structure pair:
# w1sp1r3 vs d1r4 (same NV=5 -- is the 5th view an sp doc or a review?).
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
ROWS = [(a, a.replace("wcle_", "").replace("_icetf", "")) for a, _c, _e in FLASH]
ROWS += [("__ref_i2ce_512", "i2ce @512 (= d1r3 baseline, 2000ep)"),
         ("__ref_nodoc_512", "nodoc @512 (4R+0D, 1000ep! budget mismatch)")]
_REF = {"__ref_i2ce_512": "w9_wcle_i2ce_icetf",
        "__ref_nodoc_512": "w9_wcle_nodoc_i2ce_icetf"}


def _row(lab, nm):
    zb = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    zp = Path(OUT_DIR) / f"zs_traj_{nm}_fp.json"
    ft = Path(OUT_DIR) / f"ft4var_{nm}_fp_best.json"
    if zb.exists():
        d = json.loads(zb.read_text())
        m4z = np.mean([d["nm_" + v] for v in VORD])
        line = (f"{lab:38s} ZSbest@ep{d['best_ep']:>4}(val) "
                + " ".join(f"{v[:3]}:{d['nm_' + v]:.3f}" for v in VORD)
                + f" m4z:{m4z:.3f} tag:{d['tag_neutral']:.3f}/{d['tag_noname']:.3f}")
    elif zp.exists():
        tr = json.loads(zp.read_text())
        eps = sorted(tr, key=lambda k: int(k[2:]))
        pk = max(eps, key=lambda k: tr[k]["nm_neutral"])
        line = (f"{lab:38s} ZS test-peak*@{pk[2:]:>4} neu {tr[pk]['nm_neutral']:.3f}"
                f" non {tr[pk]['nm_noname']:.3f}")
    else:
        return f"{lab:38s} (pending)"
    if ft.exists():
        d2 = json.loads(ft.read_text())
        m4 = np.mean([np.mean([x[v]["h1"] for x in d2["per_seed"]]) for v in VORD])
        line += f" | FT m4 {m4:.3f}"
    return line


for arm, lab in ROWS:
    nm = _REF.get(arm, f"w9_{arm}")
    print(_row(lab, nm))


In [ ]:
# AUTO-STOP removed in the release build: stopping the machine is cloud-
# provider tooling, not part of the experiment. All results are already on
# the shared volume when the run cells finish.
print("run complete -- results are in", OUT_DIR)